# الفصل 33: RLHF وPPO بعمق

> PEFT (الفصول 30-32) تُجيب عن سؤال "كيف نُكيِّف النموذج بكفاءة؟"، لكنها لا تُجيب عن سؤال أهم: كيف نجعل النموذج يفعل ما يريده الإنسان فعلًا، وليس فقط ما يتوقعه إحصائيًا من البيانات؟ هذا هو سؤال المحاذاة...

- **الجزء:** 7 — التخصيص الفعّال والمحاذاة
- **الفصل:** 33
- **المستوى:** خبير
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 32: QLoRA والتكميم أثناء التدريب
- **الوسوم:** RLHF، PPO، Reward Model، KL Penalty

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch33-rlhf-ppo.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-07-peft-alignment%2Fch33-rlhf-ppo.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-07-peft-alignment/ch33-rlhf-ppo.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch33-rlhf-ppo.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch33-rlhf-ppo.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch33-rlhf-ppo.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **خلايا ثقيلة:** قد تستغرق خلايا التدريب/التحميل وقتًا طويلًا. شغّلها خلية بخية، وقلّل `batch_size` أو استخدم نموذجًا أصغر عند الحاجة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch33-rlhf-ppo.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-07-peft-alignment%2Fch33-rlhf-ppo.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-07-peft-alignment/ch33-rlhf-ppo.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch33-rlhf-ppo.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch33-rlhf-ppo.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch33-rlhf-ppo.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> PEFT (الفصول 30-32) تُجيب عن سؤال "كيف نُكيِّف النموذج بكفاءة؟"، لكنها لا تُجيب عن سؤال أهم: كيف نجعل النموذج يفعل ما يريده الإنسان فعلًا، وليس فقط ما يتوقعه إحصائيًا من البيانات؟ هذا هو سؤال المحاذاة...

> **بيانات الفصل**
>
> - **الجزء:** 7 — التخصيص الفعّال والمحاذاة
> - **المستوى:** خبير
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 32: QLoRA والتكميم أثناء التدريب


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم المرحلة 2: تدريب نموذج المكافأة (Reward Model)
- فهم المرحلة 3: PPO لتحسين السياسة (Policy Optimization)
- فهم RLHF


## 1. لماذا يهمّك هذا الموضوع؟

PEFT (الفصول 30-32) تُجيب عن سؤال "كيف نُكيِّف النموذج بكفاءة؟"، لكنها لا تُجيب عن سؤال أهم: **كيف نجعل النموذج يفعل ما يريده الإنسان فعلًا، وليس فقط ما يتوقعه إحصائيًا من البيانات؟** هذا هو سؤال **المحاذاة (Alignment)**. RLHF هي التقنية التي حوّلت GPT-3 (نموذج خام يُكمل النصوص) إلى ChatGPT (مساعد يتبع التعليمات ويتجنب الضرر). فهمها ضروري لفهم كيف تُبنى كل المساعدات الذكية الحديثة.


## 2. الخريطة الذهنية


```text
خط أنابيب RLHF الكلاسيكي (OpenAI InstructGPT, 2022):

المرحلة 1: SFT (Supervised Fine-Tuning)
   نموذج أساسي + بيانات تعليمات بشرية عالية الجودة → نموذج يتبع التعليمات
                            │
                            ▼
المرحلة 2: تدريب نموذج المكافأة (Reward Model, RM)
   جمع ترتيبات تفضيل بشرية (Human Preference Rankings)
   → تدريب RM للتنبؤ بـ "أي استجابة يُفضّلها الإنسان؟"
                            │
                            ▼
المرحلة 3: التحسين عبر RL (عادة PPO)
   استخدام RM كـ "دالة مكافأة" لتحسين سياسة النموذج (Policy)
   مع قيد KL-Divergence لمنع الانحراف المفرط عن SFT
```


## 3. الشرح المتدرج

### 🟢 طبقة الحدس

تخيّل أنك تُدرِّب كلبًا. لا يمكنك أن تكتب "دالة خسارة" رياضية دقيقة تصف "السلوك الجيد"، لكن يمكنك أن تقول "أحسنت!" أو "لا" عند كل سلوك. RLHF يفعل الشيء نفسه مع LLM: بدلًا من كتابة دالة خسارة صريحة لـ"الاستجابة الجيدة" (وهو أمر شبه مستحيل رياضيًا)، نجمع تفضيلات البشر (هذه الاستجابة أفضل من تلك)، ونُدرِّب **نموذجًا منفصلًا (Reward Model)** ليتعلّم تقليد هذه التفضيلات، ثم نستخدم هذا النموذج كـ"مُدرِّب" يُوجِّه LLM الرئيسي عبر التعلّم المعزّز.

### 🟡 طبقة الآلية — المرحلة 1: SFT

هذه مرحلة تم شرحها بالتفصيل في الفصل 23 (Post-training pipeline)، لكن نُذكّر بأهميتها هنا كنقطة انطلاق: نُدرِّب النموذج الأساسي (Base Model) على أزواج (تعليمة، استجابة مثالية) مكتوبة أو مُراجَعة بشريًا، باستخدام Cross-Entropy Loss القياسية، لإنتاج نموذج "يفهم" صيغة اتباع التعليمات.

### 🟡 المرحلة 2: تدريب نموذج المكافأة (Reward Model)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RewardModel(nn.Module):
    """
    نموذج المكافأة: يأخذ نفس معمارية LLM الأساسي (عادة نفس SFT model)
    لكن نستبدل رأس اللغة (LM Head) برأس انحدار قياسي (scalar head)
    يُخرِج رقمًا واحدًا يُمثِّل "جودة" الاستجابة الكاملة.
    """
    def __init__(self, base_transformer: nn.Module, d_model: int):
        super().__init__()
        self.transformer = base_transformer  # عادة مُهيَّأ من نموذج SFT
        self.reward_head = nn.Linear(d_model, 1)

    def forward(self, input_ids: torch.Tensor,
                attention_mask: torch.Tensor) -> torch.Tensor:
        hidden_states = self.transformer(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state  # (batch, seq_len, d_model)

        # أخذ تمثيل آخر رمز فعلي (غير padding) لكل تسلسل
        seq_lengths = attention_mask.sum(dim=1) - 1
        batch_indices = torch.arange(input_ids.shape[0])
        last_token_hidden = hidden_states[batch_indices, seq_lengths]

        reward = self.reward_head(last_token_hidden)  # (batch, 1)
        return reward.squeeze(-1)  # (batch,)

def bradley_terry_reward_loss(reward_chosen: torch.Tensor,
                                reward_rejected: torch.Tensor) -> torch.Tensor:
    """
    دالة خسارة نموذج المكافأة المبنية على نموذج Bradley-Terry الإحصائي
    لاحتمالية التفضيل الزوجي:

        P(y_w ≻ y_l | x) = σ(r(x, y_w) - r(x, y_l))

    حيث y_w هي الاستجابة المُفضَّلة (winner) وy_l هي المرفوضة (loser).
    نُعظِّم log-likelihood هذا الاحتمال، أو بشكل مكافئ، نُصغِّر:
    """
    loss = -F.logsigmoid(reward_chosen - reward_rejected).mean()
    return loss

class PreferenceDataCollector:
    """
    محاكاة عملية جمع بيانات التفضيل البشري، كما في منهجية
    InstructGPT: يُعرَض على المُقيِّم البشري عدة استجابات (k=4 إلى 9)
    لنفس التعليمة، ويُرتِّبها من الأفضل إلى الأسوأ.
    من الترتيب الكلي، نستخرج أزواج (chosen, rejected) للتدريب.
    """
    @staticmethod
    def rankings_to_pairs(ranked_responses: list) -> list:
        """
        تحويل ترتيب كلي (مثال: [r1 > r3 > r2 > r4]) إلى كل الأزواج الممكنة
        (C(k,2) زوجًا)، كما فعلت ورقة InstructGPT لزيادة كفاءة البيانات.
        """
        pairs = []
        for i in range(len(ranked_responses)):
            for j in range(i + 1, len(ranked_responses)):
                pairs.append({
                    "chosen": ranked_responses[i],
                    "rejected": ranked_responses[j],
                })
        return pairs

def train_reward_model_step(reward_model: RewardModel, optimizer,
                              chosen_batch: dict, rejected_batch: dict):
    """خطوة تدريب واحدة كاملة لنموذج المكافأة."""
    optimizer.zero_grad()

    reward_chosen = reward_model(
        chosen_batch["input_ids"], chosen_batch["attention_mask"]
    )
    reward_rejected = reward_model(
        rejected_batch["input_ids"], rejected_batch["attention_mask"]
    )

    loss = bradley_terry_reward_loss(reward_chosen, reward_rejected)
    loss.backward()
    optimizer.step()

    # مقياس دقة مفيد للمراقبة: هل RM يُرتِّب الأزواج بشكل صحيح؟
    accuracy = (reward_chosen > reward_rejected).float().mean()
    return {"loss": loss.item(), "accuracy": accuracy.item()}


### 🔴 المرحلة 3: PPO لتحسين السياسة (Policy Optimization)

PPO (Proximal Policy Optimization) هو خوارزمية التعلّم المعزّز المُستخدَمة في RLHF الكلاسيكي. الفكرة الجوهرية: نُحسِّن سياسة النموذج (policy = π، أي توزيع احتمالات توليد الكلمة التالية) لتعظيم المكافأة المتوقعة من RM، **مع قيد صارم** يمنعها من الانحراف بعيدًا جدًا عن نموذج SFT الأصلي (حتى لا "تخدع" RM بإيجاد ثغرات، ظاهرة تُعرف بـ Reward Hacking).


In [ ]:
class PPOTrainerForLLM:
    """
    تطبيق مبسّط تعليميًا لحلقة تدريب PPO المُستخدَمة في RLHF.
    يتضمن أربعة نماذج يجب تشغيلها معًا:
    1. Policy Model (π_θ) — النموذج الذي نُحسِّنه (مُهيَّأ من SFT)
    2. Reference Model (π_ref) — نسخة مجمّدة من SFT (لحساب KL)
    3. Reward Model (RM) — مُجمَّد، يُقيِّم جودة الاستجابات
    4. Value Model (V) — يُقدِّر "القيمة المتوقعة" لحالة معينة (Critic)
    """
    def __init__(self, policy_model, reference_model, reward_model,
                 value_model, tokenizer, kl_coef: float = 0.1,
                 clip_epsilon: float = 0.2):
        self.policy = policy_model
        self.reference = reference_model  # مُجمَّد تمامًا، لا تدرجات
        self.reward_model = reward_model  # مُجمَّد تمامًا
        self.value_model = value_model
        self.tokenizer = tokenizer
        self.kl_coef = kl_coef
        self.clip_epsilon = clip_epsilon

        for p in self.reference.parameters():
            p.requires_grad = False
        for p in self.reward_model.parameters():
            p.requires_grad = False

    def compute_rewards_with_kl_penalty(self, prompts, responses,
                                          response_logprobs, ref_logprobs):
        """
        المكافأة الفعلية المُستخدَمة في PPO ليست فقط مخرج RM،
        بل RM مطروحًا منها عقوبة KL-divergence عن السياسة المرجعية.
        هذا يمنع النموذج من الانحراف الجذري بعيدًا عن سلوك SFT
        الآمن والمعقول سعيًا وراء "خداع" نموذج المكافأة:

            R(x,y) = RM(x,y) - β · KL(π_θ(y|x) || π_ref(y|x))

        حيث β هو kl_coef، وKL هنا تُقدَّر تجريبيًا كفرق log-probabilities.
        """
        full_sequences = [p + r for p, r in zip(prompts, responses)]
        inputs = self.tokenizer(full_sequences, return_tensors="pt",
                                  padding=True, truncation=True)

        with torch.no_grad():
            rm_scores = self.reward_model(
                inputs["input_ids"], inputs["attention_mask"]
            )

        # تقدير KL لكل رمز: log π_θ(token) - log π_ref(token)
        kl_per_token = response_logprobs - ref_logprobs
        kl_penalty = self.kl_coef * kl_per_token.sum(dim=-1)

        total_reward = rm_scores - kl_penalty
        return total_reward, kl_penalty

    def compute_advantages_gae(self, rewards, values, gamma=0.99, lam=0.95):
        """
        Generalized Advantage Estimation (GAE): تقدير "الميزة" (Advantage)
        لكل خطوة زمنية (رمز)، يُوازن بين الانحياز والتباين في تقدير
        مدى "جودة" إجراء مُعيَّن مقارنة بالمتوسط المتوقع.

            δ_t = r_t + γ·V(s_{t+1}) - V(s_t)
            A_t = δ_t + (γλ)·δ_{t+1} + (γλ)²·δ_{t+2} + ...
        """
        advantages = torch.zeros_like(rewards)
        last_gae = 0
        for t in reversed(range(len(rewards))):
            next_value = values[t + 1] if t < len(rewards) - 1 else 0
            delta = rewards[t] + gamma * next_value - values[t]
            last_gae = delta + gamma * lam * last_gae
            advantages[t] = last_gae
        returns = advantages + values
        return advantages, returns

    def ppo_clipped_objective(self, old_logprobs, new_logprobs, advantages):
        """
        دالة الهدف المُقيَّدة لـ PPO (الابتكار الجوهري للخوارزمية):
        تمنع التحديثات الكبيرة جدًا للسياسة في خطوة واحدة عبر
        "قصّ" نسبة الاحتمالات الجديدة/القديمة ضمن [1-ε, 1+ε]،
        مما يضمن استقرار التدريب (Proximal = "قريب من السياسة القديمة").

            L^CLIP(θ) = E[min(r_t(θ)·A_t, clip(r_t(θ), 1-ε, 1+ε)·A_t)]

            حيث r_t(θ) = π_θ(a_t|s_t) / π_θ_old(a_t|s_t)
        """
        ratio = torch.exp(new_logprobs - old_logprobs)  # نسبة الاحتمالات

        unclipped = ratio * advantages
        clipped = torch.clamp(
            ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon
        ) * advantages

        # نأخذ min لضمان أننا لا نُحسِّن الهدف أكثر من اللازم
        # (pessimistic bound يضمن استقرار تحديثات السياسة)
        policy_loss = -torch.min(unclipped, clipped).mean()
        return policy_loss

    def training_step(self, prompts: list):
        """خطوة PPO كاملة واحدة (مُبسَّطة تعليميًا)."""
        # 1. توليد استجابات من السياسة الحالية
        responses, response_logprobs = self._generate_with_logprobs(prompts)

        # 2. حساب log-probs من النموذج المرجعي لنفس الاستجابات (لـ KL)
        with torch.no_grad():
            ref_logprobs = self._compute_logprobs(
                self.reference, prompts, responses
            )

        # 3. حساب المكافأة مع عقوبة KL
        rewards, kl_penalty = self.compute_rewards_with_kl_penalty(
            prompts, responses, response_logprobs, ref_logprobs
        )

        # 4. حساب القيم (Critic) والميزات (Advantages)
        values = self.value_model(prompts, responses)
        advantages, returns = self.compute_advantages_gae(rewards, values)

        # 5. عدة تكرارات PPO على نفس الدفعة (عادة 4 epochs داخلية)
        for _ in range(4):
            new_logprobs = self._compute_logprobs(
                self.policy, prompts, responses
            )
            policy_loss = self.ppo_clipped_objective(
                response_logprobs.detach(), new_logprobs, advantages.detach()
            )
            value_loss = F.mse_loss(
                self.value_model(prompts, responses), returns.detach()
            )
            total_loss = policy_loss + 0.5 * value_loss
            total_loss.backward()
            # optimizer.step() ...

        return {
            "policy_loss": policy_loss.item(),
            "mean_reward": rewards.mean().item(),
            "mean_kl": kl_penalty.mean().item(),
        }

    def _generate_with_logprobs(self, prompts):
        raise NotImplementedError("تفاصيل التوليد الفعلي تعتمد على المكتبة")

    def _compute_logprobs(self, model, prompts, responses):
        raise NotImplementedError("حساب log P(token) لكل رمز في الاستجابة")


> ⚠️ **لماذا نحتاج 4 نماذج محمّلة في الذاكرة معًا؟** هذا هو السبب الرئيسي وراء **تعقيد وتكلفة RLHF/PPO العملية**: Policy + Reference + Reward + Value، أي ما يُعادل تقريبًا 4× حجم نموذج واحد في أفضل الحالات. هذا بالضبط ما حفّز تطوير بدائل أبسط مثل DPO (الفصل التالي).

---


## 4. مثال محلول خطوة بخطوة

**المسألة**: احسب خسارة Bradley-Terry لنموذج مكافأة بسيط مُعطى درجاته على 3 أزواج تفضيل.


In [ ]:
reward_pairs = [
    {"chosen": 2.3, "rejected": 0.8},   # RM واثق وصحيح
    {"chosen": 1.1, "rejected": 1.4},   # RM مُخطئ (الترتيب معكوس!)
    {"chosen": 0.2, "rejected": -0.1},  # RM صحيح لكن بثقة منخفضة
]

for i, pair in enumerate(reward_pairs):
    r_chosen = torch.tensor(pair["chosen"])
    r_rejected = torch.tensor(pair["rejected"])
    loss = -F.logsigmoid(r_chosen - r_rejected)
    correct = r_chosen > r_rejected
    print(f"زوج {i+1}: loss={loss.item():.4f}, "
          f"ترتيب صحيح؟ {correct.item()}")

# زوج 1: loss=0.1690, ترتيب صحيح؟ True
# زوج 2: loss=0.8544, ترتيب صحيح؟ False   <- خسارة أعلى بكثير لأن الترتيب خاطئ
# زوج 3: loss=0.5981, ترتيب صحيح؟ True    <- صحيح لكن الفارق صغير فالخسارة متوسطة


---


## 5. نصائح وحيل

| النصيحة | التفصيل |
|---|---|
| ابدأ RM من نموذج SFT نفسه | وليس من النموذج الأساسي الخام — يُسرّع التقارب ويُحسّن الجودة |
| راقب دقة RM على بيانات تحقق محجوزة | إذا كانت أقل من ~65-70% فالبيانات أو RM يحتاجان مراجعة |
| اضبط kl_coef بعناية | قيمة منخفضة جدًا → Reward Hacking؛ قيمة عالية جدًا → النموذج لا يتحسّن فعليًا |
| استخدم Early Stopping في RLHF | راقب "Reward-KL trade-off curve" وأوقف عند أول علامات انحراف غريب في السلوك |


## 6. أخطاء شائعة ومصائد

- **Reward Hacking**: النموذج يتعلّم "خداع" RM (مثلًا توليد استجابات طويلة جدًا لأن RM يُحابي الطول) دون تحسين حقيقي للجودة — يُكتشف عبر تقييم بشري دوري وليس فقط الاعتماد على مخرجات RM.
- **عدم تجميد Reference Model**: نسيان `requires_grad=False` يُؤدي لتسرّب تدرجات وتعطّل حساب KL بشكل صحيح.
- **بيانات تفضيل غير متناسقة بين المُقيِّمين (Low Inter-Annotator Agreement)**: تؤدي لتدريب RM على "ضوضاء" بدلًا من إشارة حقيقية؛ القياس عبر Krippendorff's alpha أو Cohen's Kappa ضروري قبل الوثوق بالبيانات.


## 7. جدول مقارنة

| النموذج | مُدرَّب؟ | الغرض |
|---|---|---|
| Policy (π_θ) | ✅ نعم (الهدف الرئيسي) | توليد الاستجابات، يتحسّن بـ PPO |
| Reference (π_ref) | ❌ مُجمَّد | مرجع لحساب KL-penalty، يمنع الانحراف المفرط |
| Reward Model (RM) | ❌ مُجمَّد (دُرِّب مسبقًا في المرحلة 2) | تقييم جودة الاستجابات |
| Value Model (Critic) | ✅ نعم | تقدير القيمة المتوقعة لتقليل تباين تقدير Advantage |


## 8. ملخص أفضل الممارسات

- جمع بيانات تفضيل بشري عالي الجودة هو **العامل الأهم** في نجاح RLHF بأكمله — جودة RM محدودة بجودة البيانات.
- استخدم Bradley-Terry Loss القياسية لتدريب RM.
- راقب دقة RM، ومتوسط المكافأة، ومتوسط KL عبر خطوات PPO باستمرار.
- اعتبر RLHF/PPO "المعيار الذهبي" تاريخيًا، لكن افحص دائمًا البدائل الأبسط (الفصل التالي) أولًا نظرًا لتعقيد التطبيق العملي العالي.


## 9. أسئلة مقابلات

**س: لماذا نحتاج عقوبة KL-divergence في PPO بدلًا من الاعتماد فقط على مخرج Reward Model؟**

**ج**: بدون قيد KL، السياسة قد "تنجرف" بعيدًا جدًا عن التوزيع الذي دُرِّب عليه RM أصلًا، فتجد **ثغرات** (Reward Hacking) يستغلها RM بشكل خاطئ لإعطاء مكافآت عالية رغم أن المخرجات الفعلية رديئة أو غير متماسكة لغويًا (RM لم يُقيَّم أصلًا على هذا النوع من المخرجات خارج توزيعه التدريبي). عقوبة KL تُبقي السياسة الجديدة "قريبة" إحصائيًا من سلوك SFT المعقول والآمن، وبذلك تُقيّد مساحة البحث لمنع الاستغلال المرضي لدالة المكافأة.

**س: ما الفرق بين الـ Reward Model والـ Value Model (Critic) في RLHF؟**

**ج**: Reward Model يُقيِّم **الاستجابة الكاملة بأكملها** مرة واحدة (مكافأة واحدة scalar لكل تسلسل كامل، نادرًا ما تُعطى على مستوى كل رمز)، ويبقى **مُجمَّدًا** طوال مرحلة PPO لأنه دُرِّب مسبقًا على بيانات التفضيل. Value Model (Critic) يُقدِّر، **لكل خطوة زمنية (كل رمز)**، "القيمة المتوقعة" للعائد الكلي المستقبلي بدءًا من تلك النقطة، ويُدرَّب **بشكل مستمر أثناء PPO** (بخسارة MSE قياسية مقابل العوائد الفعلية المُحسَبة)، وهدفه تقليل تباين تقدير Advantage عبر GAE، وليس تقييم الجودة النهائية مباشرة.


## 10. تمرين عملي

نفّذ `train_reward_model_step` كاملًا على مجموعة بيانات تفضيل عربية صغيرة (يمكن بناؤها يدويًا: 50 زوج تعليمة/استجابتين مُرتَّبتين)، وقِس دقة الترتيب (`reward_chosen > reward_rejected`) على بيانات تحقق محجوزة.


## 11. مراجع للتعمّق

- Ouyang et al. (2022) — *Training Language Models to Follow Instructions with Human Feedback* (InstructGPT)
- Schulman et al. (2017) — *Proximal Policy Optimization Algorithms*
- Christiano et al. (2017) — *Deep Reinforcement Learning from Human Preferences*
- Bradley & Terry (1952) — النموذج الإحصائي الأصلي للتفضيلات الزوجية


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — التخصيص الفعّال والمحاذاة / الفصل 33: RLHF وPPO بعمق.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
